# DASH Experiment Results Explorer

Lightweight notebook for loading and visualizing results generated by `run_experiments.py`.  
No heavy computation here — just load JSON + plot.

**Prerequisites:** Run the experiment script first:
```bash
python run_experiments.py                           # all experiments
python run_experiments.py --experiments linear_sweep # or pick one
```

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

RESULTS = Path('../results')
TABLES = RESULTS / 'tables'
FIGURES = RESULTS / 'figures'

def load(name):
    """Load a results JSON, returning None if it doesn't exist yet."""
    path = TABLES / f'{name}.json'
    if not path.exists():
        print(f'⚠ {path} not found — run the corresponding experiment first.')
        return None
    with open(path) as f:
        return json.load(f)

def show_figure(name):
    """Display a saved figure if it exists."""
    path = FIGURES / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'⚠ {path} not found.')

# Show which results are available
available = sorted(p.stem for p in TABLES.glob('*.json')) if TABLES.exists() else []
print(f'Available results: {available or "(none — run experiments first)"}')

---
## 1. Synthetic Linear Sweep

In [ ]:
sweep = load('synthetic_linear_sweep')
if sweep:
    rows = []
    for rho, methods in sorted(sweep.items(), key=lambda x: float(x[0])):
        for method, m in methods.items():
            rows.append({
                'rho': float(rho),
                'Method': method,
                'Stability': m['stability'],
                'Stability CI': f"[{m['stability_ci_lo']:.3f}, {m['stability_ci_hi']:.3f}]"
                    if 'stability_ci_lo' in m else '',
                'DGP Agreement': m.get('accuracy_mean'),
                'Equity (CV)': m['equity_mean'],
                'RMSE': m.get('rmse_mean', np.nan),
            })
    df_sweep = pd.DataFrame(rows)
    display(df_sweep.pivot_table(index='Method', columns='rho', values='Stability').round(4))
    print('\nFull table:')
    display(df_sweep.round(4))


In [ ]:
# Auto-generated figures from the script
show_figure('correlation_sweep.png')
show_figure('bar_chart_rho09.png')

In [ ]:
# Custom plot: stability comparison at high correlation
if sweep and '0.9' in sweep:
    methods = list(sweep['0.9'].keys())
    stab = [sweep['0.9'][m]['stability'] for m in methods]

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#2ecc71' if 'DASH' in m else '#95a5a6' for m in methods]
    ax.barh(methods, stab, color=colors, edgecolor='k', linewidth=0.5)
    ax.set_xlabel('Importance Stability (Pairwise Spearman)')
    ax.set_title('Stability at ρ = 0.9')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

---
## 2. Nonlinear Sweep

In [ ]:
nl = load('nonlinear_sweep')
if nl:
    rows = []
    for rho, methods in sorted(nl.items(), key=lambda x: float(x[0])):
        for method, m in methods.items():
            rows.append({
                'rho': float(rho),
                'Method': method,
                'Stability': m['stability'],
                'Equity (CV)': m['equity_mean'],
            })
    df_nl = pd.DataFrame(rows)
    display(df_nl.pivot_table(index='Method', columns='rho', values='Stability').round(4))

---
## 2.5 Overlapping Correlation Structure


In [ ]:
ol = load('overlapping')
if ol:
    rows = []
    for method, m in ol.items():
        rows.append({
            'Method': method,
            'Stability': m['stability'],
            'DGP Agreement': m.get('accuracy_mean'),
            'Equity (CV)': m.get('equity_mean'),
            'RMSE': m.get('rmse_mean'),
        })
    display(pd.DataFrame(rows).set_index('Method').round(4))


---
## 3. Table 2 Baselines (ρ = 0.9)

In [ ]:
t2 = load('table2_baselines')
if t2:
    rows = []
    for method, m in t2.items():
        row = {
            'Method': method,
            'Stability': m['stability'],
            'DGP Agreement': m.get('accuracy_mean'),
            'Equity (CV)': m['equity_mean'],
        }
        if 'stability_ci_lo' in m:
            row['Stability CI'] = f"[{m['stability_ci_lo']:.3f}, {m['stability_ci_hi']:.3f}]"
        rows.append(row)
    df_t2 = pd.DataFrame(rows).set_index('Method')
    display(df_t2.round(4))


---
## 4. Real-World Benchmarks

In [ ]:
for dataset, json_name in [('California Housing', 'california_housing'),
                            ('Breast Cancer', 'breast_cancer'),
                            ('Superconductor', 'superconductor')]:
    data = load(json_name)
    if not data:
        continue
    print(f'\n=== {dataset} ===')
    rows = []
    for method, m in data.items():
        row = {
            'Method': method,
            'Stability': m['stability'],
        }
        if 'stability_ci_lo' in m:
            row['Stability CI'] = f"[{m['stability_ci_lo']:.3f}, {m['stability_ci_hi']:.3f}]"
        if 'rmse_mean' in m:
            row['RMSE'] = f"{m['rmse_mean']:.4f} \u00b1 {m['rmse_std']:.4f}"
        if 'ablation_mean' in m:
            row['Ablation'] = f"{m['ablation_mean']:.4f} \u00b1 {m['ablation_std']:.4f}"
        rows.append(row)
    display(pd.DataFrame(rows).set_index('Method'))


In [ ]:
# IS plots and disagreement maps from the script
show_figure('is_plot_california.png')
show_figure('is_plot_breast_cancer.png')
show_figure('disagreement_breast_cancer.png')

---
## 5. Epsilon Sensitivity

In [ ]:
eps = load('epsilon_sensitivity')
if eps:
    rows = []
    for e, m in sorted(eps.items(), key=lambda x: float(x[0])):
        rows.append({
            'Epsilon': float(e),
            'Stability': m['stability'],
            'Accuracy': float(np.mean(m['acc_runs'])),
            'Models Passing': f"{np.mean(m['n_passing']):.0f} ± {np.std(m['n_passing']):.0f}",
            'K_eff': f"{np.mean(m['k_eff']):.0f} ± {np.std(m['k_eff']):.0f}",
        })
    display(pd.DataFrame(rows).set_index('Epsilon'))

    # Quick plot
    eps_vals = sorted(float(e) for e in eps.keys())
    def _eps_key(eps_dict, val):
        for k in [str(val), f"{val:.2f}", f"{val}"]:
            if k in eps_dict:
                return k
        return str(val)
    stabs = [eps[_eps_key(eps, e)]['stability'] for e in eps_vals]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(eps_vals, stabs, 'o-', color='#2ecc71', linewidth=2, markersize=8)
    ax.set_xlabel('Epsilon (ε)')
    ax.set_ylabel('Stability')
    ax.set_title('Stability vs Performance Filter Threshold')
    plt.tight_layout()
    plt.show()

---
## 6. Ablation Studies

In [ ]:
abl = load('ablation')
if abl:
    for rho_key in sorted(abl.keys(), key=float):
        print(f'\n=== ρ = {rho_key} ===')
        for param, values in abl[rho_key].items():
            rows = []
            for val, m in sorted(values.items(), key=lambda x: float(x[0])):
                rows.append({
                    param: float(val),
                    'Stability': m['stability'],
                    'Accuracy': m['accuracy_mean'],
                })
            display(pd.DataFrame(rows).set_index(param).round(4))

In [ ]:
# Ablation line plots (one subplot per parameter, lines per rho)
if abl:
    params = list(next(iter(abl.values())).keys())
    rho_keys = sorted(abl.keys(), key=float)
    rho_colors = {r: c for r, c in zip(rho_keys, ['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])}

    fig, axes = plt.subplots(1, len(params), figsize=(5 * len(params), 4), sharey=True)
    if len(params) == 1:
        axes = [axes]

    for ax, param in zip(axes, params):
        for rho_key in rho_keys:
            vals_dict = abl[rho_key][param]
            xs = sorted(float(v) for v in vals_dict.keys())
            ys = [vals_dict[str(x) if str(x) in vals_dict else str(int(x))]['stability'] for x in xs]
            ax.plot(xs, ys, 'o-', label=f'ρ={rho_key}', color=rho_colors[rho_key], linewidth=2)
        ax.set_xlabel(param)
        ax.set_title(f'Stability vs {param}')
        ax.legend(fontsize=9)

    axes[0].set_ylabel('Stability')
    fig.suptitle('Ablation Studies', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 6.5 Variance Decomposition


In [ ]:
vd = load('variance_decomposition')
if vd:
    rows = []
    for cond, methods in vd.items():
        for method, metrics in methods.items():
            rows.append({
                'Condition': cond,
                'Method': method,
                'Stability': metrics['stability'],
            })
    display(pd.DataFrame(rows).pivot_table(index='Method', columns='Condition', values='Stability').round(4))


---
## 7. Cross-Experiment Summary

Quick comparison of DASH (MaxMin) stability across all experiments.

In [ ]:
summary_rows = []

# Synthetic sweep at rho=0.9
if sweep and '0.9' in sweep and 'DASH (MaxMin)' in sweep['0.9']:
    m = sweep['0.9']['DASH (MaxMin)']
    summary_rows.append({
        'Experiment': 'Linear (ρ=0.9)',
        'Stability': m['stability'],
        'DGP Agreement': m.get('accuracy_mean'),
    })

if nl and '0.9' in nl and 'DASH (MaxMin)' in nl['0.9']:
    m = nl['0.9']['DASH (MaxMin)']
    summary_rows.append({
        'Experiment': 'Nonlinear (ρ=0.9)',
        'Stability': m['stability'],
    })

for name, json_name in [('California', 'california_housing'),
                         ('Breast Cancer', 'breast_cancer'),
                         ('Superconductor', 'superconductor')]:
    data = load(json_name)
    if data and 'DASH (MaxMin)' in data:
        summary_rows.append({
            'Experiment': name,
            'Stability': data['DASH (MaxMin)']['stability'],
        })

if summary_rows:
    display(pd.DataFrame(summary_rows).set_index('Experiment').round(4))
else:
    print('No results available yet.')